# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanmustafa119/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content/page record for one client on one reporting date.

**Time window:** I will use the March 2026 warehouse partition for development and verification. I will avoid using the June 2026 final month for developing the label because it represents the natural outcome window.




## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

**Features:**

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_pageviews`
- `sessions_organic`

**Label / proxy:**

- A future decline indicator based on the webpage's later search-performance trend.

**Context:**

- `content_type`
- client/group information, where available, for validation and grouping.

**Excluded:**

- Future-outcome or label-derived fields, because using information from the outcome period would cause data leakage.
- Client names, raw URLs, private queries, and other sensitive identifiers.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
import os
import duckdb
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Hugging Face authentication successful.")

Hugging Face authentication successful.


In [2]:
import duckdb

con = duckdb.connect()

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

print("DuckDB HTTP support loaded.")

DuckDB HTTP support loaded.


In [3]:
con.execute("""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face token configured for DuckDB.")

Hugging Face token configured for DuckDB.


In [8]:
query1 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_pages,
    COUNT(DISTINCT report_date) AS unique_dates
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────┬──────────────┐
│ total_rows │ unique_pages │ unique_dates │
│   int64    │    int64     │    int64     │
├────────────┼──────────────┼──────────────┤
│    9841378 │       331437 │           31 │
└────────────┴──────────────┴──────────────┘

### Query 1 — Grain

The March 2026 partition contains 9,841,378 rows, 331,437 unique pages, and 31 unique report dates. This shows that the warehouse contains page-level performance records across the March reporting period. I will treat the page-performance record as the unit of analysis and use the observed warehouse grain rather than making assumptions about the data.

In [10]:
query2 = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query2)

┌───────────┬───────────────┬─────────────┐
│ row_count │ earliest_date │ latest_date │
│   int64   │     date      │    date     │
├───────────┼───────────────┼─────────────┤
│   9841378 │ 2026-03-01    │ 2026-03-31  │
└───────────┴───────────────┴─────────────┘

### Query 2 — Row count and date window

The March 2026 slice contains 9,841,378 rows. The report dates range from 2026-03-01 to 2026-03-31, confirming that the selected partition covers the full March 2026 reporting window.

In [11]:
query3 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS gsc_unavailable_or_missing
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────────────┐
│ total_rows │ gsc_available_rows │ gsc_unavailable_or_missing │
│   int64    │       int64        │           int64            │
├────────────┼────────────────────┼────────────────────────────┤
│    9841378 │            3611061 │                    6230317 │
└────────────┴────────────────────┴────────────────────────────┘

### Query 3 — GSC availability

Using `gsc_data_available IS TRUE`, 3,611,061 of the 9,841,378 March 2026 rows have GSC data available. The remaining 6,230,317 rows are unavailable or missing GSC data. Therefore, GSC availability is an important limitation when building search-performance features, and I will account for it when creating my feature frame.

### Five features

I will start with five features that are available before the decision is made:

1. `gsc_impressions` — knowable at the decision moment because it is an observed Google Search measurement.
2. `gsc_clicks` — knowable at the decision moment because it records observed Search clicks.
3. `gsc_avg_position` — knowable at the decision moment because it records observed average search position.
4. `ga4_pageviews` — knowable at the decision moment because it records observed page views.
5. `sessions_organic` — knowable at the decision moment because it records observed organic sessions.

These features are measured from the selected reporting period and are available before making a recommendation rather than being future outcomes.


In [12]:
columns = con.sql("""
DESCRIBE SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

print(columns["column_name"].tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [14]:
feature_query = """
SELECT
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    sessions_organic
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 10
"""

feature_df = con.sql(feature_query).df()

print("Feature frame shape:", feature_df.shape)
feature_df

Feature frame shape: (10, 6)


,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,sessions_organic
0,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>
5,content_36c36abc7650d7af,239,1,7.347280,<NA>,<NA>
6,content_a7da352b73b02668,191,0,7.832461,<NA>,<NA>
7,content_05434271b257bb68,55,0,3.272727,<NA>,<NA>
8,content_d056587ff7faca0c,77,0,5.636364,<NA>,<NA>
9,content_bfd1e41c2af250c8,2,0,4.500000,<NA>,<NA>


### Deliberate leakage check

To demonstrate leakage, I will create a simple future-outcome proxy from a later reporting period and intentionally include it as a feature. This information would not be available at the decision moment, so it should produce an unrealistically strong result.

After observing the effect, I will remove the leaked field and retain only features that are available at the decision moment.

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Keep only rows where the required values are available
leak_test = leak_df.dropna(
    subset=["march_clicks", "april_clicks", "future_decline"]
).copy()

X_leaked = leak_test[["march_clicks", "april_clicks"]]
y = leak_test["future_decline"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaked,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

leaked_accuracy = accuracy_score(y_test, predictions)

print("Accuracy with leaked future feature:", round(leaked_accuracy, 4))

Accuracy with leaked future feature: 0.9967


### Leakage conclusion

The accuracy reached 0.99 when `april_clicks` was included as a feature. This result is not trustworthy because April information would not be available at the March decision moment, and the future outcome was defined using April performance.

This demonstrates target leakage: future information can make a model appear much stronger than it really is.

I will therefore remove `april_clicks` from the final feature set and retain only information available at the decision moment.

In [17]:
# Remove the leaked future feature from the final feature set.
honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "sessions_organic"
]

honest_feature_df = feature_df[["content_hash_id"] + honest_features].copy()

print("Final honest feature frame shape:", honest_feature_df.shape)
honest_feature_df.head()

Final honest feature frame shape: (10, 6)


,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,sessions_organic
0,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

This data contains observed search and analytics performance, but it cannot prove that a specific content change caused performance to improve or decline.

Page histories may be unbalanced, and data availability can differ across clients and dates. In the March 2026 slice, GSC data is available for only part of the rows, so search-performance features cannot be assumed to exist for every record.

The time windows used to create features and future outcomes can also overlap, so I must avoid treating overlapping measurements as independent evidence.

Therefore, the results will be used for observed, directional, decision-support recommendations rather than causal claims.

## Self-check

Before you submit, confirm each line honestly:

- ✔️ Every section above is filled — markdown thinking AND the code that backs it
- ✔️ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✔️ No client names, URLs, or private queries anywhere
- ✔️ My claims use careful words: observed, measured, directional, decision-support
- ✔️ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.